In [ ]:
# Colab bootstrap: Daten automatisch laden
from pathlib import Path
import os, sys, zipfile, urllib.request, subprocess

GITHUB_USER = "Facuriy"
GITHUB_REPO = "agro-ai-kurs"
BRANCH = "main"
BLOCK_DIR = Path("03_pflanzenkrankheit_baseline_cnn_DE")
ZIP_NAME = "03_pflanzenkrankheit_baseline_cnn_DE.zip"
ZIP_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/raw/{BRANCH}/{ZIP_NAME}"

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    # Falls in Colab Pakete fehlen, kann die folgende Zeile aktiviert werden:
    # subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy pandas matplotlib pillow scikit-learn torch torchvision"], check=False)
    if not BLOCK_DIR.exists():
        print("Lade Unterrichtsdaten:", ZIP_URL)
        urllib.request.urlretrieve(ZIP_URL, ZIP_NAME)
        with zipfile.ZipFile(ZIP_NAME, "r") as zf:
            zf.extractall(".")
    os.chdir(BLOCK_DIR)
else:
    if Path.cwd().name != BLOCK_DIR.name and BLOCK_DIR.exists():
        os.chdir(BLOCK_DIR)

print("Arbeitsordner:", Path.cwd())
print("Daten bereit:", [p.name for p in Path.cwd().iterdir() if p.is_dir()])

In [ ]:
import os, random, math, warnings, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None; DEVICE = "cpu"

print("NumPy :", np.__version__)
print("Torch :", getattr(torch, "__version__", "NICHT installiert"))
print("Gerät :", DEVICE, "(für diese Mini-Klasse reicht die CPU!)")

In [ ]:
# ✏️ HIER ÄNDERN — zentrale Einstellungen
# ---------------------------------------------------------------
DATA_DIR   = Path("mini_disease_dataset/rgb")   # train/<klasse>/, val/..., test/...
IMG_SIZE   = 128
BATCH_SIZE = 16
# ---------------------------------------------------------------
print("Dataset-Pfad:", DATA_DIR.resolve())
print("Echtes Dataset vorhanden? ->", (DATA_DIR / "train").exists())

In [ ]:
from PIL import Image

CLASS_INFO = ["00_Boden", "01_gesund", "02_first_symptoms", "03_mittel", "04_schwer"]

def _make_patch(sev, n_levels, size=128, rng=None):
    '''Synthetischer Ausschnitt. sev=0 Boden ... sev=n_levels-1 stark nekrotisch (Demo).'''
    rng = rng if rng is not None else np.random
    t = sev / max(1, n_levels - 1)                  # 0=Boden ... 1=schwer
    base = np.zeros((size, size, 3), np.float32)
    green = [0.15, 0.6, 0.2]; brown = [0.5, 0.33, 0.12]
    g_frac = 0.0 if sev == 0 else max(0.2, 1.0 - 0.9*t)   # Anteil gesundes Grün
    for ch in range(3):
        base[..., ch] = g_frac*green[ch] + (1-g_frac)*brown[ch]
    base += rng.normal(0, 0.04, base.shape); base *= rng.uniform(0.85, 1.12)
    n_spots = 0 if sev <= 1 else int(4 + 12*t)      # braune Flecken
    for _ in range(rng.randint(max(1, n_spots), n_spots+1) if n_spots else 0):
        cy, cx = rng.randint(0, size, 2); r = rng.randint(4, 11)
        yy, xx = np.ogrid[:size, :size]; m = (yy-cy)**2 + (xx-cx)**2 <= r**2
        for ch, val in enumerate(brown): base[m, ch] = val + rng.normal(0, 0.03)
    return Image.fromarray((np.clip(base, 0, 1)*255).astype(np.uint8))

def build_demo(root, n_per=45, size=128, seed=0):
    rng = np.random.RandomState(seed)
    if root.exists(): shutil.rmtree(root)
    for sev, cls in enumerate(CLASS_INFO):
        imgs = [_make_patch(sev, len(CLASS_INFO), size, rng) for _ in range(n_per)]
        i = 0
        for split, frac in {"train":0.6, "val":0.2, "test":0.2}.items():
            d = root/split/cls; d.mkdir(parents=True, exist_ok=True)
            for _ in range(int(round(n_per*frac))):
                if i < len(imgs): imgs[i].save(d/f"{cls}_{i:03d}.png"); i += 1

if not (DATA_DIR / "train").exists():
    print(f"⚠️  Kein echtes Dataset -> erzeuge {len(CLASS_INFO)}-Klassen-DEMO ...")
    build_demo(DATA_DIR, n_per=45, size=IMG_SIZE, seed=SEED)
    print("✅ Demo-Daten erstellt.")
else:
    print("✅ Echtes Dataset gefunden.")

In [ ]:
import torch
from torchvision import datasets, transforms

tf_basic = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
train_ds = datasets.ImageFolder(DATA_DIR/"train", transform=tf_basic)
val_ds   = datasets.ImageFolder(DATA_DIR/"val",   transform=tf_basic)
test_ds  = datasets.ImageFolder(DATA_DIR/"test",  transform=tf_basic)

CLASSES = train_ds.classes              # z.B. ['00_Boden','01_gesund','02_mittel','03_schwer']
N_CLASSES = len(CLASSES)
print("Klassen:", CLASSES)
print(f"Bilder -> Training {len(train_ds)} | Validierung {len(val_ds)} | Test {len(test_ds)}")

In [ ]:
# Ein Beispiel pro Klasse zeigen (mehrere Spalten)
def show_examples(ds, n=5):
    buckets = {c: [] for c in range(N_CLASSES)}
    for img, y in ds:
        if len(buckets[y]) < n: buckets[y].append(img)
        if all(len(v) >= n for v in buckets.values()): break
    fig, axes = plt.subplots(N_CLASSES, n, figsize=(2*n, 2*N_CLASSES))
    if N_CLASSES == 1: axes = axes[None, :]
    for r in range(N_CLASSES):
        for c in range(n):
            ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([])
            if c < len(buckets[r]): ax.imshow(buckets[r][c].permute(1,2,0).numpy())
            if c == 0: ax.set_ylabel(CLASSES[r], fontsize=9, rotation=0, ha="right", va="center")
    fig.suptitle("Beispielbilder je Klasse (oben gesund/Boden → unten schwer)", fontsize=12)
    plt.tight_layout(); plt.show()

show_examples(train_ds, n=5)

In [ ]:
from collections import Counter
def counts(ds):
    c = Counter([y for _, y in ds.samples])
    return [c.get(i, 0) for i in range(N_CLASSES)]

import numpy as np
tr, va, te = counts(train_ds), counts(val_ds), counts(test_ds)
for name, cc in [("Training", tr), ("Validierung", va), ("Test", te)]:
    print(f"{name:12s}:", {CLASSES[i]: cc[i] for i in range(N_CLASSES)})

x = np.arange(N_CLASSES); w = 0.6
plt.figure(figsize=(7, 3))
plt.bar(x, tr, w, color=["#a1662f", "#4caf50", "#9acd32", "#d4a017", "#8b3a2f"][:N_CLASSES])
plt.xticks(x, CLASSES, rotation=20, ha="right"); plt.ylabel("Anzahl (Training)")
plt.title("Klassenbalance im Trainingsset"); plt.tight_layout(); plt.show()

In [ ]:
def image_features(img):
    x = img.numpy(); R, G, B = x[0], x[1], x[2]
    return [R.mean(), G.mean(), B.mean(), (2*G-R-B).mean(), ((R>G)&(G>B)).mean()]
FEATURE_NAMES = ["Rot", "Grün", "Blau", "ExG (Grünheit)", "Braun-Anteil"]

def feats(ds):
    X = np.array([image_features(i) for i, _ in ds]); y = np.array([y for _, y in ds]); return X, y
Xtr, ytr = feats(train_ds); Xte, yte = feats(test_ds)
print("Merkmals-Matrix Training:", Xtr.shape, "(Bilder × Merkmale)")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

rf = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(Xtr, ytr)
acc_base = accuracy_score(yte, rf.predict(Xte))
print(f"Baseline (Random Forest) — Test-Genauigkeit: {acc_base*100:.1f}%  "
      f"(Zufall wäre {100/N_CLASSES:.0f}%)")

fig, ax = plt.subplots(figsize=(6, 3)); order = np.argsort(rf.feature_importances_)
ax.barh(np.array(FEATURE_NAMES)[order], rf.feature_importances_[order], color="#4682b4")
ax.set_title("Wichtigkeit der Merkmale (Baseline)"); plt.tight_layout(); plt.show()

cm = confusion_matrix(yte, rf.predict(Xte))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap="Blues", colorbar=False, xticks_rotation=30)
plt.title("Baseline: Confusion-Matrix (Test)"); plt.tight_layout(); plt.show()

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class MiniCNN(nn.Module):
    def __init__(self, img_size=128, n_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        feat = img_size // 4
        self.fc1 = nn.Linear(16*feat*feat, 32)
        self.fc2 = nn.Linear(32, n_classes)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        return self.fc2(F.relu(self.fc1(x)))

model = MiniCNN(IMG_SIZE, N_CLASSES).to(DEVICE)
print(f"Mini-CNN — {sum(p.numel() for p in model.parameters()):,} Parameter, {N_CLASSES} Klassen.")

In [ ]:
# ✏️ HIER ÄNDERN — Trainings-Einstellungen
# ---------------------------------------------------------------
EPOCHS        = 12
LEARNING_RATE = 0.001
USE_AUGMENT   = True
# ---------------------------------------------------------------
aug = [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
       transforms.RandomRotation(20)] if USE_AUGMENT else []
train_ds.transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), *aug, transforms.ToTensor()])
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH_SIZE)
print(f"Augmentation: {'AN' if USE_AUGMENT else 'AUS'} | Epochen {EPOCHS} | Lernrate {LEARNING_RATE}")

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    tot, corr, ls = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb); loss = loss_fn(out, yb)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            ls += loss.item()*len(yb); corr += (out.argmax(1)==yb).sum().item(); tot += len(yb)
    return ls/tot, corr/tot

hist = {k: [] for k in ["tl", "vl", "ta", "va"]}
for ep in range(1, EPOCHS+1):
    tl, ta = run_epoch(train_loader, True); vl, va = run_epoch(val_loader, False)
    for k, v in zip(hist, [tl, vl, ta, va]): hist[k].append(v)
    print(f"Epoche {ep:2d}/{EPOCHS} | Train-Loss {tl:.3f} Acc {ta*100:4.1f}% | Val-Loss {vl:.3f} Acc {va*100:4.1f}%")
print("Fertig! ✅")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6)); ep = range(1, EPOCHS+1)
a1.plot(ep, hist["tl"], "o-", label="Training"); a1.plot(ep, hist["vl"], "s-", label="Validierung")
a1.set_title("Loss (niedriger = besser)"); a1.set_xlabel("Epoche"); a1.legend()
a2.plot(ep, [v*100 for v in hist["ta"]], "o-", label="Training"); a2.plot(ep, [v*100 for v in hist["va"]], "s-", label="Validierung")
a2.set_title("Genauigkeit %"); a2.set_xlabel("Epoche"); a2.legend()
plt.tight_layout(); plt.show()
print("👀 Training steigt stark, Validierung nicht -> Overfitting (auswendig gelernt).")

In [ ]:
model.eval(); y_true, y_pred, keep = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        pred = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        y_pred.extend(pred); y_true.extend(yb.numpy())
        for i in range(len(yb)):
            if len(keep) < 80: keep.append((xb[i], int(yb[i]), int(pred[i])))
acc_cnn = (np.array(y_true)==np.array(y_pred)).mean()
print(f"Mini-CNN — Test-Genauigkeit: {acc_cnn*100:.1f}%   (Baseline: {acc_base*100:.1f}%)")
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap="Greens", colorbar=False, xticks_rotation=30)
plt.title("Mini-CNN: Confusion-Matrix (Test)"); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import classification_report, f1_score

print("Klassifikationsbericht der Mini-CNN")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
print(f"Macro-F1: {f1_score(y_true, y_pred, average='macro'):.3f}")

In [ ]:
correct = [t for t in keep if t[1]==t[2]][:4]
wrong   = [t for t in keep if t[1]!=t[2]][:4]
def show_row(items, title, color):
    if not items: print(f"({title}: keine Beispiele)"); return
    fig, axes = plt.subplots(1, len(items), figsize=(2.6*len(items), 3.0))
    if len(items)==1: axes=[axes]
    for ax,(img,yt,yp) in zip(axes, items):
        ax.imshow(img.permute(1,2,0).numpy()); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"wahr: {CLASSES[yt]}\nModell: {CLASSES[yp]}", color=color, fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()
show_row(correct, "✅ Richtig erkannt", "green")
show_row(wrong,   "❌ Fehler des Modells", "red")

In [ ]:
from sklearn.model_selection import cross_val_score, GroupKFold
X_all = np.vstack([Xtr, Xte]); y_all = np.concatenate([ytr, yte])
rf2 = RandomForestClassifier(n_estimators=100, random_state=0)
acc_naiv  = cross_val_score(rf2, X_all, y_all, cv=5).mean()
groups = np.arange(len(y_all)) // 3                       # künstliche Gruppen (Demo)
acc_grp = cross_val_score(rf2, X_all, y_all, groups=groups, cv=GroupKFold(5)).mean()
print(f"(a) Zufälliger Split : {acc_naiv*100:.1f}%   <- kann zu OPTIMISTISCH sein")
print(f"(b) Gruppen-Split    : {acc_grp*100:.1f}%   <- EHRLICHER (keine Leakage)")

In [ ]:
def fake_ndvi(sev, n, size=128, rng=np.random):
    base_nir = 0.85 - 0.55 * (sev / max(1, n-1))    # gesund hoch -> schwer niedrig
    nir = rng.uniform(base_nir-0.08, base_nir+0.08, (size, size)); red = rng.uniform(0.15, 0.3, (size, size))
    return (nir-red)/(nir+red+1e-6)
fig, axes = plt.subplots(1, N_CLASSES, figsize=(2.6*N_CLASSES, 3))
for ax, sev in zip(axes, range(N_CLASSES)):
    im = ax.imshow(fake_ndvi(sev, N_CLASSES), cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_title(CLASSES[sev], fontsize=8); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.046, label="NDVI"); plt.show()
print("Grün = hoher NDVI (vital), Rot = niedriger NDVI (gestresst).")